# PYNQ 504 - Board to Board Communication

Learn to send messages between PYNQ boards and read sensor data!

By the end of this session, you will:
* Send messages between PYNQ boards
* Understand client-server architecture
* Share sensor data across boards

## Real-World Applications

* **Self-driving cars:** Share radar/camera data between vehicles
* **Warehouse robots:** Amazon robots communicate positions to avoid collisions  
* **Drone swarms:** Share altitude/position sensors to fly in formation
* **Smart factories:** Sensor networks monitor temperature, vibration, quality

---
## What is PYNQ P2P?

### Peer-to-Peer Communication

P2P (Peer-to-Peer) means **devices talking directly to each other**, not just to a central computer.

Think of it like:
* **Walkie-talkies** - You and your friend can talk directly
* **Text messaging** - Send messages to specific people
* **Multiplayer games** - Players communicate in real-time

### How PYNQ P2P Works

PYNQ boards use a **central server** (like a post office) to route messages:

```
Your Board                  P2P Server              Teammate's Board
   "Hi!"      ────────►    [Stores message]    ◄────────  [Checks inbox]
              ◄────────     "Message sent!"     ────────►  "Got it!"
```

**Key Features:**
* Each board gets a **unique ID** (based on MAC address - like a phone number)
* Send messages to **specific boards** by their ID
* Messages are **queued** until the recipient reads them
* **Secure** with password protection

### Why Use P2P?

**🤖 Distributed Robotics**
* Amazon warehouse: 520,000+ robots coordinating together
* Each robot knows where others are to avoid collisions
* Share sensor data between robots for better decisions

**🚗 Autonomous Vehicles**
* Self-driving cars share radar/camera data
* Vehicle-to-vehicle (V2V) communication
* Warn each other about obstacles, traffic, weather

**🏭 Smart Manufacturing**
* Factory sensors share temperature, pressure, quality data
* Machines coordinate production schedules
* Predictive maintenance alerts

**✈️ Drone Swarms**
* Multiple drones flying in formation
* Share GPS, altitude, battery status
* Coordinate search & rescue missions

### The Process:
```
1. Register with P2P server (get your unique ID)
2. Send messages to other boards (using their ID)
3. Receive messages sent to you
4. Share sensor data in real-time
```

Let's get started! 🚀

In [ ]:
# Install required libraries
!pip3 install getmac -q
!pip3 install ./pynq-p2p -q

# Import the library
import pynqp2p

print("PYNQ P2P libraries installed and imported!")

### Step 1: Register with the P2P Server

To ensure security, we use a secret passcode to register with the server. Your instructor will provide the server IP and passcode.

In [ ]:
# Register with the P2P server
# Your instructor will provide these values
p2p_ip_address = '[IP address of P2P server]'
p2p_key = '[secret passcode]'

pynqp2p.register(p2p_ip_address, p2p_key)
print("Registered with P2P server!")

### Step 2: Test the Connection

Let's verify we can communicate with the server.

In [ ]:
# Ping the server
pynqp2p.ping()
print("Connection successful!")

### Step 3: Get Your Unique ID

Every PYNQ board has a unique **MAC address** - like a serial number for network devices!

**Think of it like:**
* Your phone number (unique to you)
* Your student ID (unique in your school)
* License plate on a car (unique to that car)

No two network devices in the world have the same MAC address!

The P2P system uses this to create a unique ID for your board.

**Share this ID with your teammate** so they can send you messages!

In [ ]:
# Get your unique board ID
myid = pynqp2p.get_id()
print(f"Your board ID is: {myid}")
print("\nShare this ID with your teammate!")

### Step 4: Send a Message

Now that you have your ID and your teammate's ID, you can send messages!

In [ ]:
# Send a message to your teammate
# Replace with your teammate's actual ID
recipient_id = 'paste teammate ID here'

pynqp2p.send(recipient_id, "Hello from my PYNQ board!")
print(f"Message sent to {recipient_id}")

### Step 5: Receive Messages

Check if your teammate sent you a message! The `receive()` function retrieves the most recent message sent to you.

**Note:** Each message can only be received once - after you receive it, it's deleted from the server.

In [ ]:
# Receive a message
message = pynqp2p.receive()
print(f"Received message: {message}")

### Sending Multiple Messages

You can send multiple messages in a row. Let's send 5 messages to ourselves for practice.

In [ ]:
# Send multiple messages
for i in range(5):
    pynqp2p.send(myid, f"Test message {i}")
    
print("Sent 5 messages to yourself!")

### Receiving All Messages

Instead of calling `receive()` multiple times, we can use `receive_all()` to get all pending messages at once.

In [ ]:
# Receive all pending messages
messages = pynqp2p.receive_all()
print(f"Received {len(messages)} messages:")
for msg in messages:
    print(f"  - {msg}")

---
## Ultrasonic Sensor + P2P

**Hardware needed:** Ultrasonic sensor + Grove PMOD Adapter

![ultrasonic-sensor.png](ultrasonic-sensor.png)

![pmod.png](pmod.png)

**Goal:** Send distance measurements from your sensor to your teammate's board!

**Setup:** Attach the PMOD adapter to PMODA and connect the ultrasonic sensor to port G4.

This is what it should look like:

![Ultrasonic_connector.jpg](Ultrasonic_connector.jpg)

### Why Share Sensor Data?

In real robotics systems, **sensor fusion** combines data from multiple sources:

* **Self-driving cars**: Share radar/camera data between vehicles to detect obstacles
* **Warehouse robots**: Amazon robots communicate positions to avoid collisions  
* **Drone swarms**: Share altitude/position sensors to fly in formation
* **Smart factories**: Sensor networks monitor temperature, vibration, quality

**Your task**: One board reads a sensor, another board receives the data and could act on it!

In [ ]:
# Setup ultrasonic sensor
from pynq_peripherals import PmodGroveAdapter
from kv260 import BaseOverlay
from time import sleep

base = BaseOverlay('base.bit')
adapter = PmodGroveAdapter(base.PMODA, G4='grove_usranger')
usranger = adapter.G4

print("Ultrasonic sensor ready!")
print("Max range: 500cm (5 meters)")

In [ ]:
# Test the sensor - take 5 readings
for i in range(5):
    distance = usranger.get_distance()
    print(f'Reading {i+1}: {distance:.2f} cm')
    sleep(1)

---
## Challenges

Test your skills!

### Challenge 1: Send Sensor Readings

**Task 1:** Read distance and send it to your teammate

**Task 2:** Send 10 readings in a loop (1 per second)

**Task 3:** Receive and display your teammate's sensor data

**BONUS:** If distance < 50cm, send "CLOSE" message, else send "FAR" message

In [ ]:
# Challenge 1: Send sensor readings to your teammate
#
# Task 1: Read distance and send it
# distance = usranger.get_distance()
# pynqp2p.send(recipient_id, f"Distance: {distance:.2f} cm")
#
# Task 2: Send 10 readings in a loop
# for i in range(10):
#     distance = usranger.get_distance()
#     pynqp2p.send(recipient_id, f"{distance:.2f}")
#     sleep(1)
#
# Task 3: Receive and display your teammate's sensor data
# messages = pynqp2p.receive_all()
# for msg in messages:
#     print(f"Teammate's sensor: {msg}")
#
# BONUS: If distance < 50cm, send "CLOSE" message, else send "FAR" message

### Cleanup Sensor

In [ ]:
# Clean up the sensor overlay
del base
print("Sensor overlay released")

---
## Summary

Great work! You learned:

✓ **P2P messaging** between PYNQ boards  
✓ **Sensor data sharing** for distributed systems  
✓ **Client-server** architecture  
✓ **Real-world applications** of networked sensors

These skills power smart factories, autonomous vehicles, and IoT systems!

**Next:** Check out PYNQ 506 to control a robot arm in simulation!